# Pull every trade for one closed Polymarket market

1. Resolve a closed market by slug (Gamma API).
2. Page through all of its trades (Data API `/v2/trades`).
3. Load the trades into a DataFrame and show a few rows.

Uses the repo's read-only `polyresearch.client`, so no credentials are needed.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the repo root importable when the kernel starts in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from polyresearch.client import Client, GAMMA, pages

client = Client()

In [ ]:
# A closed market: Hornets vs Wizards, Oct 30 2026, total points over/under 219.5
SLUG = "nba-cha-was-2026-10-30-total-219pt5"

# Gamma hides closed markets from slug lookups unless closed=True is passed
(market,) = client.get(GAMMA, "/markets", slug=SLUG, closed=True)
assert market["closed"], "expected a closed market"

pd.Series({
    "question": market["question"],
    "condition_id": market["conditionId"],
    "closed": market["closed"],
    "end_date": market["endDate"],
    "outcomes": market["outcomes"],
    "final_prices": market["outcomePrices"],
    "volume_usdc": round(market["volumeNum"], 2),
})

question                           Hornets vs. Wizards: O/U 219.5
condition_id    0xbc735e93184be87286a47a5472138aba335d4864c76b...
closed                                                       True
end_date                                     2026-10-30T23:00:00Z
outcomes                                        ["Over", "Under"]
final_prices                                           ["0", "1"]
volume_usdc                                               59990.5
dtype: object

In [ ]:
# taker_only=True returns each fill once (the taker side); with False you also get the maker legs.
rows, raw_pages, complete = pages(
    client,
    "/v2/trades",
    {"condition": market["conditionId"], "taker_only": True, "limit": 1000},
    max_pages=100,
)
print(f"{len(rows):,} trades over {len(raw_pages)} pages; feed exhausted: {complete}")

253 trades over 1 pages; feed exhausted: True


In [ ]:
trades = pd.DataFrame(rows)
trades["time_utc"] = pd.to_datetime(trades["timestamp"], unit="s", utc=True)
trades["notional_usdc"] = trades["size"] * trades["price"]
trades = trades.sort_values("time_utc").reset_index(drop=True)

cols = ["time_utc", "proxy_wallet", "pseudonym", "side", "outcome",
        "size", "price", "notional_usdc", "transaction_hash"]
trades[cols].head(10)

,time_utc,proxy_wallet,pseudonym,side,outcome,size,price,notional_usdc,transaction_hash
0,2026-10-30 02:22:37+00:00,0x5268527977f700f9bf9b6d5cd843859e4e70135d,Powerless-Savannah,BUY,Under,28.000000,0.45,12.60,0xb3a69cbda7cd720e9f648270a343575fa1e3a9fe448f...
1,2026-10-30 02:22:43+00:00,0x7cef52839763912c557acfda54b4387660830a88,Mysterious-Overshoot,BUY,Under,2.595745,0.47,1.22,0x66ba805ea71f03c3a788f833716b23c8046c0d2e21f8...
2,2026-10-30 02:25:37+00:00,0xdd5ef115f801999575b03d78d0115ad8b2d59040,Smug-Variability,BUY,Under,20.000000,0.47,9.40,0x70195dfb34c20a8ada74427a87177fe89e74a72843f2...
3,2026-10-30 02:25:39+00:00,0x5268527977f700f9bf9b6d5cd843859e4e70135d,Powerless-Savannah,BUY,Under,9.000000,0.47,4.23,0xf2cbb84bb89584dcd263b7d6e269360e676b36704c7f...
4,2026-10-30 02:25:39+00:00,0xddb993dbaee76440dbb42408a1925e38b7aaedcd,Boiling-Sprat-Dwell,BUY,Under,100.000000,0.48,48.00,0x97ee37cecb7a835e1d4ea67c99f09c58567d95ad29dd...
5,2026-10-30 02:26:03+00:00,0x5268527977f700f9bf9b6d5cd843859e4e70135d,Powerless-Savannah,BUY,Under,120.000000,0.49,58.80,0x6356ac82235814c95fd6ba8b3270e6f261f8c1294d94...
6,2026-10-30 02:26:06+00:00,0x8d6c30d1ffbe79baaec8b65c79164b822965c143,Black-Keyboarding,BUY,Under,5.040000,0.50,2.52,0x95f0e1aa68169fbb70aa42d55561cd528e7d4c1dd6dd...
7,2026-10-30 02:26:07+00:00,0x68dac0822eec92435ef9ffa3304d13ea248d9a05,Bad-Buying-Hail,BUY,Under,10.060000,0.50,5.03,0x5efed5bf34712eacb37be00c60109a47d531e7d59d90...
8,2026-10-30 05:23:04+00:00,0x5268527977f700f9bf9b6d5cd843859e4e70135d,Powerless-Savannah,BUY,Over,123.000000,0.50,61.50,0x862462f4d8b78dddd13818929523c091300c2427b2a6...
9,2026-10-30 05:27:31+00:00,0x08f38148bb62214e8711095720fce0804109f168,Imaginary-Outset-Run,BUY,Over,5.098040,0.51,2.60,0xf80970e0126e6aef1bdf4efee7ca04b82b771a660e8e...


In [ ]:
# Quick sanity summary
print("first trade:", trades["time_utc"].min())
print("last trade: ", trades["time_utc"].max())
print("unique wallets:", trades["proxy_wallet"].nunique())
trades.groupby(["outcome", "side"])["notional_usdc"].agg(["count", "sum"]).round(2)

first trade: 2026-10-30 02:22:37+00:00
last trade:  2026-10-31 01:32:19+00:00
unique wallets: 88


count       sum
outcome side                 
Over    BUY      94  12231.76
        SELL      6     44.35
Under   BUY     147  18057.68
        SELL      6     71.47